<a href="https://colab.research.google.com/github/Spandana2704/DL/blob/main/WEEK13_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Implement BERT model : predicting next word in sentence, Finding missing words in sentence, review classification**

In [1]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 29.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


**Next Word Prediction**

In [4]:
from transformers import BertTokenizer, BertForMaskedLM
import torch, string

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

text = "I am going to the [MASK] to buy groceries"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
mask_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

top5 = torch.topk(logits[0, mask_index], 5, dim=1).indices[0].tolist()

for i in top5:
    word = tokenizer.decode([i]).strip()
    if word not in string.punctuation:
        print(word)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


store
market
mall
supermarket
bank


**Fill Missing Word in Sentence**

In [6]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

text = "The capital of India is [MASK] [MASK]"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
mask_positions = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

predicted_words = []

for pos in mask_positions:
    predicted_id = logits[0, pos].argmax(axis=-1)
    predicted_words.append(tokenizer.decode([predicted_id]))

print(" ".join(predicted_words))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mumbai .


**Review Classification (Sentiment Analysis)**

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("textattack/bert-base-uncased-imdb")
model = AutoModelForSequenceClassification.from_pretrained("textattack/bert-base-uncased-imdb")

text = "This product is amazing!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
pred = torch.argmax(logits, dim=1).item()

labels = ["Negative", "Positive"]
print(labels[pred])

config.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Positive


**Implement Transformer model**

In [9]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(nn.Module):
    def __init__(self, src_vocab, trg_vocab, d_model=128, heads=8, layers=2):
        super().__init__()

        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.trg_emb = nn.Embedding(trg_vocab, d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=heads,
            num_encoder_layers=layers,
            num_decoder_layers=layers
        )

        self.fc = nn.Linear(d_model, trg_vocab)

    def forward(self, src, trg):
        src = self.src_emb(src)
        trg = self.trg_emb(trg)

        src = src.permute(1, 0, 2)
        trg = trg.permute(1, 0, 2)

        out = self.transformer(src, trg)

        out = out.permute(1, 0, 2)
        return self.fc(out)


src_vocab = 100
trg_vocab = 100

model = Model(src_vocab, trg_vocab).to(device)

src = torch.randint(0, 100, (2, 10)).to(device)
trg = torch.randint(0, 100, (2, 10)).to(device)

out = model(src, trg)

print(out.shape)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


torch.Size([2, 10, 100])


In [11]:
import torch

sentences = ["i love ai", "hello world"]

vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "i":3, "love":4, "ai":5, "hello":6, "world":7}

def encode(s):
    return [1] + [vocab[w] for w in s.split()] + [2]

data = [encode(s) for s in sentences]

max_len = max(len(x) for x in data)

padded = [x + [0]*(max_len - len(x)) for x in data]

src = torch.tensor(padded)
trg = torch.tensor(padded)

print(src)

tensor([[1, 3, 4, 5, 2],
        [1, 6, 7, 2, 0]])


**Implement ViT model**

**Using PyTorch + Torchvision**

In [12]:
import torch
from torchvision.models import vit_b_16

model = vit_b_16(pretrained=True)

model.heads = torch.nn.Linear(model.heads.head.in_features, 10)

img = torch.randn(2, 3, 224, 224)

out = model(img)

print(out.shape)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:05<00:00, 59.2MB/s]


torch.Size([2, 10])


**Using Hugging Face Transformers**

In [13]:
from transformers import ViTForImageClassification, ViTImageProcessor
import torch

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=10
)

img = torch.randn(2, 3, 224, 224)

out = model(pixel_values=img)

print(out.logits.shape)

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([10])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


RuntimeError: You set `ignore_mismatched_sizes` to `False`, thus raising an error. For details look at the above report!